# 00 Smoke test: Medusae Fossae x Egypt (real data, terrain only)

Goal: one real end-to-end run. MOLA target tile + ETOPO Egypt tiles -> `run_real.py` -> `out/`.
No climate features yet, so only the terrain bars are scored. **Not a finding until validated.**

Raw downloads and results are kept on Google Drive, so a new Colab session skips the downloads.
Clear all outputs before committing this notebook (Edit -> Clear all outputs).

In [ ]:
# 1. Repo + install (re-run after every new session)
import os
if not os.path.exists('/content/earth-twin'):
    !git clone -q https://github.com/Demerdashh/earth-twin.git
%cd /content/earth-twin
!git pull -q
!pip -q install -e ".[real]"
!python tests/test_core.py

In [ ]:
# 2. Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/earth-twin')
RAW, DATA = DRIVE / 'raw', DRIVE / 'data'
for p in (RAW, DATA, DATA / 'targets'):
    p.mkdir(parents=True, exist_ok=True)
print(DRIVE)

## Mars target: MOLA MEGDR 128 px/deg, tile megt44n180hb

Covers 44N to 0 and 180E to 270E (PDS label megt44n180hb.xml): 5632 lines x 11520 samples,
SignedMSB2 (big-endian int16), metres, equirectangular, 128 px/deg. The target (11N, 159W = 201E) is inside.
We read the raw file with numpy instead of GDAL, so there are no PDS label issues.

In [ ]:
# 3. Download the MOLA tile (about 130 MB, once)
MOLA = RAW / 'megt44n180hb.img'
URL = 'https://pds-geosciences.wustl.edu/mgs/urn-nasa-pds-mgs_mola_topography_derived/meg128/megt44n180hb.img'
if not MOLA.exists() or MOLA.stat().st_size != 129_761_280:
    !wget -q -c -O "{MOLA}" "{URL}"
print(MOLA, MOLA.stat().st_size, 'bytes (expected 129761280)')

In [ ]:
# 4. Cut the Medusae Fossae target tile at 463 m x 64 px (same format as scripts/cut_target.py)
import json, numpy as np
from IPython.display import Image, display
from analoglens.geo import GlobalRaster
from analoglens.render import save_relief_png

LINES, SAMPLES = 5632, 11520
raw = np.fromfile(MOLA, dtype='>i2')
assert raw.size == LINES * SAMPLES, raw.size
mola = GlobalRaster(raw.reshape(LINES, SAMPLES), lat_top=44.0, lon_left=180.0, deg_per_px=1 / 128)
del raw

KEY = 'medusae_fossae'
cfg = json.loads(Path('config/targets.json').read_text(encoding='utf-8'))[KEY]
lat, lon = cfg['lat'], cfg['lon']          # change here to try another spot inside the formation
tile = mola.tile(lat, lon, 64, 463.0, 'mars')
print(f'valid {np.isfinite(tile).mean():.0%}, relief {np.nanmax(tile) - np.nanmin(tile):.0f} m')
np.savez_compressed(DATA / 'targets' / f'{KEY}.npz', tile=tile.astype(np.float32), key=KEY, body='mars',
                    lat=lat, lon=lon, res_m=463.0)
save_relief_png(tile, 463.0, DATA / 'targets' / f'{KEY}.png')
display(Image(str(DATA / 'targets' / f'{KEY}.png'), width=320))

**Stop and look at the PNG.** Linear, wind-aligned ridges = good. Smooth or blotchy = the
463 m grid is too coarse or interpolated here; move `lat`/`lon` inside the formation and re-run cell 4.

In [ ]:
# 5. Download ETOPO 2022 15 arc-second surface tiles for Egypt (about 30 MB each, once)
BASE = 'https://www.ngdc.noaa.gov/mgg/global/relief/ETOPO2022/data/15s/15s_surface_elev_gtif'
ETOPO = []
for name in ['ETOPO_2022_v1_15s_N30E015_surface.tif', 'ETOPO_2022_v1_15s_N30E030_surface.tif']:
    path = RAW / name
    if not path.exists():
        !wget -q -c -O "{path}" "{BASE}/{name}"
    ETOPO.append(path)
    print(path.name, path.stat().st_size // 2**20, 'MB')

In [ ]:
# 6. Cut Earth tiles over Egypt: 22-30N (N30 tiles end at 30N), 24-36E, 30 km step
!python scripts/cut_tiles.py --raster "{ETOPO[0]}" --raster "{ETOPO[1]}" \
    --bbox 22 30 24 36 --step-km 30 --res-m 463 --size 64 --prefix EG --out "{DATA}/earth_egypt.npz"

In [ ]:
# 7. Rank: descriptors baseline, then DINOv2 (GPU optional; a few thousand tiles run on CPU too)
!python scripts/run_real.py --earth "{DATA}/earth_egypt.npz" --targets "{DATA}/targets" \
    --embedder descriptors --cards 10 --out /content/out_desc
!python scripts/run_real.py --earth "{DATA}/earth_egypt.npz" --targets "{DATA}/targets" \
    --embedder dino --cards 10 --out /content/out_dino

In [ ]:
# 8. Look at the top 10 and compare the two embedders
import json
for name in ('out_desc', 'out_dino'):
    r = json.loads(Path(f'/content/{name}/rank/medusae_fossae__wind_erosion_science.json').read_text())
    print(name, 'embedder:', r['embedder'], '| synthetic:', r['synthetic'])
    for x in r['results'][:10]:
        print(f"  {x['rank']:>2} {x['tile_id']}  {x['lat']:7.3f} {x['lon']:7.3f}  score {x['score']:.3f}  dims {x['dims']}")
top = json.loads(Path('/content/out_dino/rank/medusae_fossae__wind_erosion_science.json').read_text())['results'][0]
display(Image(f"/content/out_dino/img/earth_{top['tile_id']}.png", width=320))

In [ ]:
# 9. Save results to Drive (one zip each; Drive is slow with many small files)
import shutil
for name in ('out_desc', 'out_dino'):
    shutil.make_archive(str(DRIVE / f'smoke_{name}'), 'zip', f'/content/{name}')
!ls -la "{DRIVE}"